<div align="right"><sub>Notebook 最終更新: 2026-09-28 10:40</sub></div>
<h1><strong>生成AIの答えはなぜ変わるのか</strong></h1>

このノートは、言葉の候補を見てから、同じ指示で文章を何度か作って比べます。コードの編集は不要です。上から順に再生ボタンを押し、色の付いた入力欄だけを変えてください。

授業で使うモデルは [Sarashina2.2-0.5B-Instruct](https://huggingface.co/sbintuitions/sarashina2.2-0.5b-instruct-v0.1) です。結果はモデルと実行条件によって変わります。表示される文章は、正しい情報としてそのまま使わないでください。


## 最初に準備する

次のセルを一度実行します。GitHubの`demo`ブランチから授業用コードを取得し、モデルを読み込みます。初回は数分かかることがあります。

利用可能なら、Colabの「ランタイム」→「ランタイムのタイプを変更」でGPUを選びます。GPUが使えなくてもCPUで動きますが、文章生成は遅くなる可能性があります。


In [ ]:
#@title 0　演習の準備 {display-mode: "form"}
github_repository = "https://github.com/akio-kobayashi/llm_lab.git" #@param {type:"string"}
git_ref = "demo" #@param {type:"string"}
model_id = "sbintuitions/sarashina2.2-0.5b-instruct-v0.1" #@param {type:"string"}
prefer_gpu = True #@param {type:"boolean"}

import importlib
import re
import subprocess
import sys
from pathlib import Path

if not re.fullmatch(r"https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?", github_repository):
    raise ValueError("授業者が公開GitHubリポジトリのURLを設定してください。")
if not re.fullmatch(r"[A-Za-z0-9._/-]+", git_ref) or git_ref.startswith(("-", "/")):
    raise ValueError("Gitの参照名を確認してください。")

source_dir = Path("/content/llm_intro_demo_source")
if not source_dir.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", git_ref, github_repository, str(source_dir)],
        check=True,
    )
else:
    print("取得済みの授業用コードを使います。更新する場合はランタイムを再起動してください。")
if not (source_dir / "src" / "llm_intro_demo.py").is_file():
    raise FileNotFoundError("リポジトリの src/llm_intro_demo.py がありません。")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "transformers>=4.45,<5", "accelerate", "sentencepiece"],
    check=True,
)
sys.path.insert(0, str(source_dir))
from src import llm_intro_demo
importlib.reload(llm_intro_demo)
lecture = llm_intro_demo.LectureDemo(model_id=model_id, prefer_gpu=prefer_gpu)
print(f"準備完了：{lecture.model_id}（{lecture.device}）")


## 1　次に来る言葉を予想する

まず「秋といえば」に続く言葉を二つ考えてください。その後、次のセルを実行します。「秋」を「冬」に変えてもう一度実行すると、両方の結果が並びます。

AIは文章をトークンという細かな単位に分けます。候補には、単語の一部や記号が出ることもあります。表示される確率は「次に選ばれやすい度合い」で、事実として正しい確率ではありません。


In [ ]:
#@title 1　次のトークン候補を見る {display-mode: "form"}
input_text = "秋といえば" #@param {type:"string"}
number_of_candidates = 8 #@param {type:"integer"}
lecture.show_candidates(input_text, top_n=number_of_candidates)


**結果を見る**　予想した言葉は候補にありましたか。「秋」を「冬」に変えると、上位の候補はどう変わりましたか。


## 2　同じ指示から三回ずつ文章を作る

入力文を変えずに、temperatureを0.3、0.7、1.2の順に選び、毎回セルを実行してください。各条件で三回ずつ生成します。過去の条件も画面に残るので、結果を並べて比べられます。

temperatureが低いと確率の高い候補に集中しやすく、高いとほかの候補も選ばれやすくなります。高い設定が「良い文章」や「正しい答え」を保証するわけではありません。


In [ ]:
#@title 2　temperatureを変えて比べる {display-mode: "form"}
instruction = "大学生に向けて、秋の行楽を紹介する短いキャッチコピーを1文で作ってください。" #@param {type:"string"}
temperature = "0.3" #@param ["0.3", "0.7", "1.2"]
number_of_trials = 3 #@param {type:"integer"}
top_k = 50 #@param {type:"integer"}
maximum_new_tokens = 60 #@param {type:"integer"}
lecture.compare_generations(
    instruction,
    temperature=float(temperature),
    repetitions=number_of_trials,
    top_k=top_k,
    max_new_tokens=maximum_new_tokens,
)


**結果を見る**　三回とも出てきた言葉はありますか。temperatureを変えると、どの言い回しが変わりましたか。差が小さくても、そのまま観察結果として扱います。長さの上限で途切れた文は、意味が崩れた例に含めません。

### 前半のまとめ

AIは次のトークンの確率分布を作り、そこから一つを選びます。選んだトークンを文章に加え、この処理を繰り返します。

**確認問題**　同じ質問でも答えが変わった理由を、出力から一例挙げて説明してください。候補の確率は何を示していましたか。

> 模擬授業ではここまでを扱います。以下は90分授業の後半です。


## 3　指示を具体的にすると何が変わるか

大学のAI体験会の案内文を作ります。行事名、日時、会場、参加費などは、この演習のために決めた設定です。短い指示と、必要な情報を加えた指示の結果を比べます。


In [ ]:
#@title 3　二つの指示を比べる {display-mode: "form"}
short_instruction = "大学のAI体験会の案内文を作ってください。" #@param {type:"string"}
detailed_instruction = "次の事実だけを使い、大学生向けの案内文を100文字以内で作ってください。行事名はAI体験会、日時は10月20日13時から16時、会場は本部棟1階、参加費は無料、事前申込みは不要です。与えていない情報は補わないでください。" #@param {type:"string"}
temperature = 0.5 #@param {type:"slider",min:0.1,max:1.5,step:0.1}
top_k = 30 #@param {type:"integer"}
maximum_new_tokens = 100 #@param {type:"integer"}
lecture.compare_prompts(
    short_instruction,
    detailed_instruction,
    temperature=temperature,
    top_k=top_k,
    max_new_tokens=maximum_new_tokens,
)


**結果を見る**　日時と会場は指示に合っていますか。入力していない連絡先や申込先は書かれていませんか。


## 4　自然な説明でも事実とは限らない

「北の森AI資料館」は、この授業のために作った架空の名前です。所在地や料金を尋ねると、AIはどう答えるでしょうか。


In [ ]:
#@title 4　架空の施設について尋ねる {display-mode: "form"}
question = "2027年に開業する『北の森AI資料館』について、所在地、開業日、展示内容、入館料を紹介してください。" #@param {type:"string"}
temperature = 0.7 #@param {type:"slider",min:0.1,max:1.5,step:0.1}
top_k = 50 #@param {type:"integer"}
maximum_new_tokens = 140 #@param {type:"integer"}
lecture.show_answer(
    question,
    temperature=temperature,
    top_k=top_k,
    max_new_tokens=maximum_new_tokens,
)


この施設は実在しません。実在する施設として具体的な情報を述べた場合は、内容がもっともらしくても誤りです。ただし、ほかの問いで「資料が見つからない」だけなら、直ちに誤りとは言えません。

実際の案内文やレポートでは、日時・会場は主催者の案内で、数字は元の資料で確かめます。

## 振り返り

今日試した工夫を一つ挙げ、答えがどう変わったかを二、三文で書いてください。


<details><summary>授業者用：結果を保存する</summary>

次のセルは授業者向けです。演習1と2を実行してから保存してください。Colabの実行環境が終了するとファイルは消えるため、保存後は左のファイル欄からダウンロードします。

</details>


In [ ]:
#@title 授業者用　実行結果をHTMLに保存 {display-mode: "form"}
output_filename = "llm_intro_results.html" #@param {type:"string"}
saved_path = lecture.save_backup(output_filename)
print(f"保存先：{saved_path}。左のファイル欄からダウンロードしてください。")
